# Parte 1 — MDP + Q-Learning tabular (despacho de batería / BESS)

**Problema:** desplazamiento de demanda (*load shifting*), no especulación financiera.
La batería compra energía a la red cuando el precio de bolsa es bajo y cubre demanda
propia (costo evitado) cuando es alto, respetando las restricciones físicas del BESS.

Flujo del notebook:
1. Cargar precios horarios: XM (`pydataxm`) → CSV plan B → serie simulada.
2. Dividir los días cronológicamente en entrenamiento (80%) y prueba (20%).
3. Entrenar el agente `QLearningAgent` 500 episodios (cada episodio = un día al azar).
4. Evaluar sobre los días de prueba el **ahorro acumulado en COP** contra la Línea Base.

In [ ]:
from __future__ import annotations

import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.part1_rl.environment import BatteryEnv, Accion
from src.part1_rl.agent import QLearningAgent

SEMILLA = 42
EPISODIOS = 500
FRACCION_TRAIN = 0.8

# Fuente de datos: True intenta XM / plan B; False usa directamente la serie simulada.
USAR_XM = False
FECHA_INICIO, FECHA_FIN = date(2024, 1, 1), date(2024, 6, 30)

# Parámetros físicos: capacidad 10 kWh -> 4 niveles de 2.5 kWh.
# Con ENERGIA_POR_NIVEL_KWH = 1.0 la recompensa es exactamente ±precio (COP/kWh).
ENERGIA_POR_NIVEL_KWH = 1.0

## 1. Datos de precio horario

`PrecBolsNaci` de XM llega en formato ancho (`Date`, `Values_Hour01` … `Values_Hour24`).
`a_serie_horaria` lo convierte a una fila por hora y descarta días incompletos.
Si XM y el plan B fallan, se usa una serie simulada con valle de madrugada y pico vespertino.

In [ ]:
def a_serie_horaria(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza datos de XM a columnas (timestamp, precio_cop_kwh), días completos."""
    columnas_hora = sorted(c for c in df.columns if str(c).startswith("Values_Hour"))
    if columnas_hora:  # formato ancho de pydataxm
        largo = df.melt(id_vars="Date", value_vars=columnas_hora,
                        var_name="col", value_name="precio_cop_kwh")
        # Values_Hour01 corresponde al periodo 00:00-01:00.
        hora = largo["col"].str[-2:].astype(int) - 1
        largo["timestamp"] = pd.to_datetime(largo["Date"]) + pd.to_timedelta(hora, unit="h")
    elif {"timestamp", "precio_cop_kwh"} <= set(df.columns):
        largo = df.copy()
        largo["timestamp"] = pd.to_datetime(largo["timestamp"])
    else:
        raise ValueError(f"Formato de precios no reconocido: {list(df.columns)}")

    largo = largo[["timestamp", "precio_cop_kwh"]].copy()
    largo["precio_cop_kwh"] = pd.to_numeric(largo["precio_cop_kwh"], errors="coerce")
    largo = largo.dropna().sort_values("timestamp").drop_duplicates("timestamp")
    horas_por_dia = largo.groupby(largo["timestamp"].dt.date)["timestamp"].transform("size")
    return largo[horas_por_dia == 24].reset_index(drop=True)


def simular_precios(dias: int = 120, semilla: int = SEMILLA) -> pd.DataFrame:
    """Precios horarios simulados (COP/kWh): patrón diario + nivel diario + ruido."""
    rng = np.random.default_rng(semilla)
    horas = np.arange(24)
    patron = (250 + 120 * np.sin((horas - 8) / 24 * 2 * np.pi)
              + 90 * np.exp(-((horas - 19) ** 2) / 8.0))
    nivel_diario = 1 + 0.25 * np.sin(np.arange(dias) / 30 * 2 * np.pi)  # efecto "hidrológico"
    precios = np.concatenate([
        np.clip(patron * nivel_diario[d] + rng.normal(0, 25, 24), 50, None) for d in range(dias)
    ])
    ts = pd.date_range("2026-01-01", periods=dias * 24, freq="h")
    return pd.DataFrame({"timestamp": ts, "precio_cop_kwh": precios})


def cargar_precios() -> tuple[pd.DataFrame, str]:
    if USAR_XM:
        try:
            from src.data.download_xm import obtener_precio_bolsa
            df = a_serie_horaria(obtener_precio_bolsa(FECHA_INICIO, FECHA_FIN))
            if len(df) >= 48:
                return df, "XM"
            print("[WARN] Datos XM insuficientes; se usa la serie simulada.")
        except Exception as exc:  # noqa: BLE001 - cualquier fallo activa la simulación
            print(f"[WARN] No se pudieron cargar datos XM ({exc}); se usa la serie simulada.")
    return simular_precios(), "simulado"


df_precios, FUENTE = cargar_precios()
df_precios["fecha"] = df_precios["timestamp"].dt.date
df_precios["hora"] = df_precios["timestamp"].dt.hour
n_dias = df_precios["fecha"].nunique()
print(f"Fuente: {FUENTE} | días: {n_dias} | horas: {len(df_precios)}")
print(df_precios["precio_cop_kwh"].describe().round(1))

In [ ]:
perfil = df_precios.groupby("hora")["precio_cop_kwh"].agg(["mean", "std"])
plt.figure(figsize=(9, 3.5))
plt.plot(perfil.index, perfil["mean"], marker="o", label="Promedio")
plt.fill_between(perfil.index, perfil["mean"] - perfil["std"], perfil["mean"] + perfil["std"],
                 alpha=0.2, label="±1 desviación")
plt.axvline(2, color="tab:green", ls="--", label="Baseline: cargar 02:00")
plt.axvline(19, color="tab:red", ls="--", label="Baseline: descargar 19:00")
plt.title(f"Precio de bolsa por hora ({FUENTE})")
plt.xlabel("Hora del día"); plt.ylabel("COP/kWh"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. División entrenamiento / prueba y entornos

La división es cronológica para no evaluar con días que el agente ya vio.
Los cuartiles de precio se calculan solo con los días de entrenamiento y se
reutilizan en el entorno de prueba.

In [ ]:
fechas = sorted(df_precios["fecha"].unique())
corte = max(1, int(len(fechas) * FRACCION_TRAIN))
if corte >= len(fechas):
    raise ValueError("Se necesitan al menos 2 días para separar entrenamiento y prueba.")
fechas_train, fechas_test = fechas[:corte], fechas[corte:]

precios_train = df_precios[df_precios["fecha"].isin(fechas_train)]["precio_cop_kwh"].to_numpy()
precios_test = df_precios[df_precios["fecha"].isin(fechas_test)]["precio_cop_kwh"].to_numpy()

env_train = BatteryEnv(precios_train, soc_inicial=0, penalizacion_invalida=-1000.0,
                       energia_por_nivel_kwh=ENERGIA_POR_NIVEL_KWH, aleatorio=True, semilla=SEMILLA)
env_test = BatteryEnv(precios_test, soc_inicial=0, penalizacion_invalida=-1000.0,
                      energia_por_nivel_kwh=ENERGIA_POR_NIVEL_KWH,
                      umbrales_precio=env_train.umbrales_precio, aleatorio=False)

print(f"Días entrenamiento: {env_train.n_dias} | días prueba: {env_test.n_dias}")
print("Umbrales de cuartil (p25, p50, p75):", env_train.umbrales_precio.round(1))

## 3. Entrenamiento (500 episodios)

`alpha = 0.7`: con solo 500 episodios, una tasa baja (0.1) no alcanza a propagar el
valor de la descarga de la tarde hacia la compra de madrugada, y el agente converge a
"Mantener" siempre (ahorro 0). En un barrido de 10 semillas sobre los días de prueba,
el ahorro medio fue ≈ 0.3k COP con `alpha=0.1`, ≈ 8.1k con 0.5, ≈ 10.7k con 0.7 y
≈ 13.6k con 0.9, frente a ≈ 5.5k de la Línea Base. Se elige 0.7 por menor varianza.

In [ ]:
agente = QLearningAgent(
    n_acciones=BatteryEnv.N_ACCIONES,
    alpha=0.7,
    gamma=0.99,
    epsilon_inicial=1.0,
    epsilon_final=0.05,
    episodios_decaimiento=int(EPISODIOS * 0.8),  # últimos 20% casi greedy
    semilla=SEMILLA,
)
retornos = agente.entrenar(env_train, episodios=EPISODIOS)
print(f"Episodios: {len(retornos)} | epsilon final: {agente.epsilon:.3f} | "
      f"estados visitados: {len(agente.q_table)} de {24 * 4 * 5}")

serie = pd.Series(retornos)
plt.figure(figsize=(9, 3.5))
plt.plot(serie, alpha=0.3, label="Retorno por episodio")
plt.plot(serie.rolling(20, min_periods=1).mean(), label="Media móvil (20)")
plt.title("Curva de aprendizaje (incluye penalizaciones por acciones inválidas)")
plt.xlabel("Episodio"); plt.ylabel("Recompensa del episodio")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Artefactos para la herramienta `simular_despacho` del agente LLM (Parte 2):
#   - precios de los días de prueba (la herramienta simula sobre estos días)
#   - umbrales de cuartil del entrenamiento (sin ellos la Q-Table no es reutilizable)
DIR_PROCESADO = RAIZ / "data" / "processed"
ruta_eval = DIR_PROCESADO / "precios_evaluacion.csv"
(df_precios[df_precios["fecha"].isin(fechas_test)][["timestamp", "precio_cop_kwh"]]
 .to_csv(ruta_eval, index=False))

ruta_q = agente.guardar(
    DIR_PROCESADO / "q_table.json",
    metadatos={
        "fuente_datos": FUENTE,
        "umbrales_precio": env_train.umbrales_precio.tolist(),
        "soc_inicial": env_train.soc_inicial,
        "penalizacion_invalida": env_train.penalizacion_invalida,
        "energia_por_nivel_kwh_entrenamiento": ENERGIA_POR_NIVEL_KWH,
        "episodios": EPISODIOS,
        "fechas_entrenamiento": [str(fechas_train[0]), str(fechas_train[-1])],
        "fechas_evaluacion": [str(fechas_test[0]), str(fechas_test[-1])],
        "archivo_precios_evaluacion": ruta_eval.name,
    },
)
print(f"Q-Table guardada en {ruta_q}")
print(f"Precios de evaluación guardados en {ruta_eval}")

## 4. Evaluación en días de prueba: ahorro acumulado (COP)

Cada política recorre todos los días de prueba en orden, sin exploración.
El ahorro usa el flujo de caja real (`info["flujo_caja_cop"]`), sin la penalización artificial.

In [ ]:
def evaluar(env: BatteryEnv, selector) -> pd.DataFrame:
    """Ejecuta `selector(estado) -> accion` sobre todos los días del entorno."""
    filas: list[dict] = []
    for dia in range(env.n_dias):
        estado = env.reset(dia=dia)
        terminado = False
        while not terminado:
            estado, _, terminado, info = env.step(selector(estado))
            filas.append(info)
    df = pd.DataFrame(filas)
    df["accion"] = df["accion"].map(lambda a: Accion(a).name)
    df["ahorro_acumulado_cop"] = df["flujo_caja_cop"].cumsum()
    return df


df_ql = evaluar(env_test, lambda s: agente.seleccionar_accion(s, explorar=False))
df_base = evaluar(env_test, BatteryEnv.naive_policy)

resumen = pd.DataFrame({
    "Q-Learning": [df_ql["ahorro_acumulado_cop"].iloc[-1], (~df_ql["valida"]).sum(),
                   (df_ql["accion"] == "CARGAR").sum(), (df_ql["accion"] == "DESCARGAR").sum()],
    "Línea Base": [df_base["ahorro_acumulado_cop"].iloc[-1], (~df_base["valida"]).sum(),
                   (df_base["accion"] == "CARGAR").sum(), (df_base["accion"] == "DESCARGAR").sum()],
}, index=["Ahorro total (COP)", "Acciones inválidas", "Cargas", "Descargas"])
resumen

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_ql["ahorro_acumulado_cop"].to_numpy(), label="Q-Learning")
plt.plot(df_base["ahorro_acumulado_cop"].to_numpy(), label="Línea Base (02:00 / 19:00)")
plt.axhline(0, color="gray", lw=0.8)
plt.title(f"Ahorro acumulado en COP sobre {env_test.n_dias} días de prueba ({FUENTE})")
plt.xlabel("Hora de simulación"); plt.ylabel("Ahorro acumulado (COP)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

total_ql = df_ql["ahorro_acumulado_cop"].iloc[-1]
total_base = df_base["ahorro_acumulado_cop"].iloc[-1]
print(f"Ahorro Q-Learning : {total_ql:,.1f} COP")
print(f"Ahorro Línea Base : {total_base:,.1f} COP")
print(f"Diferencia        : {total_ql - total_base:,.1f} COP")

In [ ]:
# Política aprendida: acción greedy por hora y cuartil de precio, con SoC = 50%.
nombres = {0: "M", 1: "C", 2: "D"}
politica = pd.DataFrame(
    [[nombres[int(np.argmax(agente.q_table[(h, c, 2)]))] if (h, c, 2) in agente.q_table else "·"
      for h in range(24)] for c in range(4)],
    index=[f"Q{c + 1}" for c in range(4)], columns=range(24),
)
print("M=Mantener, C=Cargar, D=Descargar, ·=estado no visitado (SoC 50%)")
politica